# Experiment: Variables acústicas relevantes

**Pregunta.** ¿Qué características (MFCC, log-mel, ZCR, centroide, bandwidth, rolloff) separan sirena de tráfico y, en menor medida, los tipos etiquetados?

**Criterio de éxito.** Resumen por clase, figuras comparativas y una conclusión operativa: log-mel 2D para CNN, MFCC como baseline compacto.


In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

SEED = 7

# Local: data/raw/<corpus>
# Kaggle (Add Input clásico): /kaggle/input/<slug>
# Kaggle (datasets/user): /kaggle/input/datasets/<user>/<slug>/<slug>
IS_KAGGLE = Path("/kaggle/input").exists()

if IS_KAGGLE:
    DATA_ROOT = Path("/kaggle/input")
    FIGURES_DIR = Path("/kaggle/working/reports/figures")
    TABLES_DIR = Path("/kaggle/working/reports/tables")
else:
    here = Path.cwd().resolve()
    REPO_ROOT = here if (here / "data" / "raw").exists() else here.parent
    DATA_ROOT = REPO_ROOT / "data" / "raw"
    FIGURES_DIR = REPO_ROOT / "reports" / "figures"
    TABLES_DIR = REPO_ROOT / "reports" / "tables"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

CORPUS_SLUGS = {
    "sirennet": ("sirennet",),
    "lssiren": ("lssiren",),
    "urbansound8k": ("urbansound8k",),
}


def is_kaggle() -> bool:
    return IS_KAGGLE


def figures_dir() -> Path:
    return FIGURES_DIR


def tables_dir() -> Path:
    return TABLES_DIR


def resolve_corpus(name: str) -> Path | None:
    candidates: list[Path] = []
    for slug in CORPUS_SLUGS[name]:
        candidates.append(DATA_ROOT / slug)
        datasets = DATA_ROOT / "datasets"
        if datasets.exists():
            for user_dir in datasets.iterdir():
                if not user_dir.is_dir():
                    continue
                candidates.append(user_dir / slug)
                candidates.append(user_dir / slug / slug)
    existing = [path for path in candidates if path.exists()]
    if not existing:
        return None
    return max(existing, key=lambda path: len(path.parts))


@dataclass(frozen=True)
class CorpusPaths:
    sirennet: Path | None
    lssiren: Path | None
    urbansound8k: Path | None

    def available(self) -> dict[str, Path]:
        found = {
            "sirennet": self.sirennet,
            "lssiren": self.lssiren,
            "urbansound8k": self.urbansound8k,
        }
        return {key: path for key, path in found.items() if path is not None}


def corpus_paths() -> CorpusPaths:
    return CorpusPaths(
        sirennet=resolve_corpus("sirennet"),
        lssiren=resolve_corpus("lssiren"),
        urbansound8k=resolve_corpus("urbansound8k"),
    )


print("kaggle:", IS_KAGGLE)
print("DATA_ROOT:", DATA_ROOT)
print("FIGURES_DIR:", FIGURES_DIR)
print("TABLES_DIR:", TABLES_DIR)
print("available:", list(corpus_paths().available()))
SEED


kaggle: False
DATA_ROOT: /home/jeancdevx/dev/doppler/doppler-ml/data/raw
FIGURES_DIR: /home/jeancdevx/dev/doppler/doppler-ml/reports/figures
TABLES_DIR: /home/jeancdevx/dev/doppler/doppler-ml/reports/tables
available: ['sirennet', 'lssiren', 'urbansound8k']


7

In [2]:
# Lectura de audio y descriptores
"""Lectura de audio y características con librosa, o scipy si no está disponible."""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np

try:
    import librosa

    HAS_LIBROSA = True
except Exception:  # pragma: no cover - entorno sin ruedas de librosa
    HAS_LIBROSA = False
    librosa = None  # type: ignore

from scipy.io import wavfile
from scipy.signal import get_window, spectrogram, stft

try:
    import soundfile as sf

    HAS_SOUNDFILE = True
except Exception:
    HAS_SOUNDFILE = False
    sf = None  # type: ignore


TARGET_SR = 22050


@dataclass
class AudioClip:
    y: np.ndarray
    sr: int


def load_audio(path: str, sr: int = TARGET_SR, duration: float | None = None) -> AudioClip:
    if HAS_LIBROSA:
        y, out_sr = librosa.load(path, sr=sr, mono=True, duration=duration)
        return AudioClip(y=np.asarray(y, dtype=np.float32), sr=out_sr)

    if HAS_SOUNDFILE:
        y, file_sr = sf.read(path, always_2d=False)
        y = np.asarray(y, dtype=np.float32)
        if y.ndim > 1:
            y = y.mean(axis=1)
        if duration is not None:
            y = y[: int(duration * file_sr)]
        if sr and file_sr != sr:
            duration_s = len(y) / float(file_sr)
            n_out = max(1, int(duration_s * sr))
            x_old = np.linspace(0.0, duration_s, num=len(y), endpoint=False)
            x_new = np.linspace(0.0, duration_s, num=n_out, endpoint=False)
            y = np.interp(x_new, x_old, y).astype(np.float32)
            file_sr = sr
        return AudioClip(y=y, sr=int(file_sr))

    file_sr, data = wavfile.read(path)
    y = np.asarray(data, dtype=np.float32)
    if y.ndim > 1:
        y = y.mean(axis=1)
    max_abs = np.max(np.abs(y)) or 1.0
    if max_abs > 1.5:
        y = y / 32768.0
    if duration is not None:
        y = y[: int(duration * file_sr)]
    if sr and file_sr != sr:
        duration_s = len(y) / file_sr
        n_out = int(duration_s * sr)
        x_old = np.linspace(0.0, duration_s, num=len(y), endpoint=False)
        x_new = np.linspace(0.0, duration_s, num=n_out, endpoint=False)
        y = np.interp(x_new, x_old, y).astype(np.float32)
        file_sr = sr
    return AudioClip(y=y, sr=file_sr)


def duration_seconds(path: str) -> float:
    return float(wav_probe(path)["duration_s"])


def wav_probe(path: str) -> dict:
    """Metadatos baratos sin resamplear."""
    if HAS_SOUNDFILE:
        info = sf.info(path)
        return {
            "sr": int(info.samplerate),
            "n_channels": int(info.channels),
            "n_samples": int(info.frames),
            "duration_s": float(info.duration),
            "dtype": str(info.subtype),
        }
    try:
        sr, data = wavfile.read(path)
        n_channels = 1 if np.asarray(data).ndim == 1 else np.asarray(data).shape[1]
        n_samples = int(np.asarray(data).shape[0])
        return {
            "sr": int(sr),
            "n_channels": int(n_channels),
            "n_samples": n_samples,
            "duration_s": n_samples / float(sr),
            "dtype": str(np.asarray(data).dtype),
        }
    except Exception:
        if HAS_LIBROSA:
            y, sr = librosa.load(path, sr=None, mono=False)
            y = np.asarray(y)
            n_channels = 1 if y.ndim == 1 else y.shape[0]
            n_samples = y.shape[-1]
            return {
                "sr": int(sr),
                "n_channels": int(n_channels),
                "n_samples": int(n_samples),
                "duration_s": n_samples / float(sr),
                "dtype": str(y.dtype),
            }
        raise


def log_mel_spectrogram(clip: AudioClip, n_mels: int = 64, n_fft: int = 1024, hop: int = 256) -> np.ndarray:
    if HAS_LIBROSA:
        S = librosa.feature.melspectrogram(y=clip.y, sr=clip.sr, n_mels=n_mels, n_fft=n_fft, hop_length=hop)
        return librosa.power_to_db(S, ref=np.max)
    f, t, Sxx = spectrogram(clip.y, fs=clip.sr, nperseg=n_fft, noverlap=n_fft - hop, window="hann")
    # Aproximación: filtro triangular en Hz de Mel.
    mel_f = _hz_to_mel(f)
    edges = np.linspace(mel_f.min(), mel_f.max(), n_mels + 2)
    mels = np.zeros((n_mels, Sxx.shape[1]), dtype=np.float32)
    for i in range(n_mels):
        lo, mid, hi = edges[i], edges[i + 1], edges[i + 2]
        w = np.zeros_like(mel_f)
        left = np.logical_and(mel_f >= lo, mel_f <= mid)
        right = np.logical_and(mel_f >= mid, mel_f <= hi)
        if np.any(left):
            w[left] = (mel_f[left] - lo) / max(mid - lo, 1e-8)
        if np.any(right):
            w[right] = (hi - mel_f[right]) / max(hi - mid, 1e-8)
        mels[i] = w @ Sxx
    mels = np.maximum(mels, 1e-10)
    return 10.0 * np.log10(mels / np.max(mels))


def mfcc(clip: AudioClip, n_mfcc: int = 13) -> np.ndarray:
    if HAS_LIBROSA:
        return librosa.feature.mfcc(y=clip.y, sr=clip.sr, n_mfcc=n_mfcc)
    log_mel = log_mel_spectrogram(clip, n_mels=40)
    # DCT tipo II sobre el eje mel.
    n_mels, n_frames = log_mel.shape
    n = np.arange(n_mels)
    k = np.arange(n_mfcc)[:, None]
    dct = np.cos(np.pi * k * (2 * n + 1) / (2.0 * n_mels))
    return dct @ log_mel


def spectral_centroid(clip: AudioClip) -> np.ndarray:
    if HAS_LIBROSA:
        return librosa.feature.spectral_centroid(y=clip.y, sr=clip.sr)[0]
    f, _, Zxx = stft(clip.y, fs=clip.sr, nperseg=1024)
    mag = np.abs(Zxx)
    denom = np.sum(mag, axis=0) + 1e-10
    return (f[:, None] * mag).sum(axis=0) / denom


def spectral_bandwidth(clip: AudioClip) -> np.ndarray:
    if HAS_LIBROSA:
        return librosa.feature.spectral_bandwidth(y=clip.y, sr=clip.sr)[0]
    f, _, Zxx = stft(clip.y, fs=clip.sr, nperseg=1024)
    mag = np.abs(Zxx)
    denom = np.sum(mag, axis=0) + 1e-10
    centroid = (f[:, None] * mag).sum(axis=0) / denom
    var = ((f[:, None] - centroid) ** 2 * mag).sum(axis=0) / denom
    return np.sqrt(var)


def spectral_rolloff(clip: AudioClip, roll_percent: float = 0.85) -> np.ndarray:
    if HAS_LIBROSA:
        return librosa.feature.spectral_rolloff(y=clip.y, sr=clip.sr, roll_percent=roll_percent)[0]
    f, _, Zxx = stft(clip.y, fs=clip.sr, nperseg=1024)
    mag = np.abs(Zxx)
    csum = np.cumsum(mag, axis=0)
    thresh = roll_percent * (csum[-1] + 1e-10)
    idx = np.argmax(csum >= thresh, axis=0)
    return f[idx]


def zero_crossing_rate(clip: AudioClip, frame_length: int = 2048, hop: int = 512) -> np.ndarray:
    if HAS_LIBROSA:
        return librosa.feature.zero_crossing_rate(clip.y, frame_length=frame_length, hop_length=hop)[0]
    y = clip.y
    n = 1 + max(0, (len(y) - frame_length) // hop)
    out = np.zeros(n, dtype=np.float32)
    for i in range(n):
        frame = y[i * hop : i * hop + frame_length]
        out[i] = np.mean(np.abs(np.diff(np.signbit(frame))))
    return out


def _hz_to_mel(hz: np.ndarray) -> np.ndarray:
    return 2595.0 * np.log10(1.0 + hz / 700.0)


def stft_db(clip: AudioClip, n_fft: int = 1024, hop: int = 256) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    if HAS_LIBROSA:
        S = np.abs(librosa.stft(clip.y, n_fft=n_fft, hop_length=hop))
        db = librosa.amplitude_to_db(S, ref=np.max)
        freqs = librosa.fft_frequencies(sr=clip.sr, n_fft=n_fft)
        times = librosa.frames_to_time(np.arange(db.shape[1]), sr=clip.sr, hop_length=hop)
        return freqs, times, db
    window = get_window("hann", n_fft)
    f, t, Zxx = stft(clip.y, fs=clip.sr, window=window, nperseg=n_fft, noverlap=n_fft - hop)
    mag = np.abs(Zxx)
    db = 20.0 * np.log10(np.maximum(mag, 1e-10) / (np.max(mag) + 1e-10))
    return f, t, db


## Plan

- Hipótesis 1: sirena vs tráfico se separa bien en centroide/rolloff y en los MFCC bajos (tonalidad periódica).
- Hipótesis 2: ambulance/police/firetruck se solapan más que sirena vs traffic.
- Muestra: hasta 40 clips por clase de sireNNet (reproducible con SEED=7).
- Métricas: media ± std de descriptores; mapa log-mel y curva MFCC de un ejemplo.


In [3]:
import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
rng = np.random.default_rng(SEED)
FIG = figures_dir()
TAB = tables_dir()
print("librosa:", HAS_LIBROSA)

inv_path = TAB / "file_inventory.csv"
inventory = pd.read_csv(inv_path) if inv_path.exists() else pd.DataFrame()
from pathlib import Path as _P
if not inventory.empty:
    inventory = inventory[inventory["path"].map(lambda p: _P(str(p)).exists())]
inventory.groupby(["corpus", "label"]).size() if not inventory.empty else "empty"


librosa: False


corpus    label    
sirennet  ambulance    400
          firetruck    400
          police       454
          traffic      421
dtype: int64

## Muestreo estratificado y extracción


In [4]:
MAX_PER_CLASS = 40
sirennet = inventory[inventory["corpus"] == "sirennet"] if not inventory.empty else pd.DataFrame()
sample_df = pd.DataFrame()

if sirennet.empty:
    print("sireNNet no disponible")
else:
    parts = []
    for label, sub in sirennet.groupby("label"):
        parts.append(sub.sample(min(MAX_PER_CLASS, len(sub)), random_state=SEED))
    sample_df = pd.concat(parts, ignore_index=True)

rows = []
for _, row in sample_df.iterrows():
    clip = load_audio(row["path"], duration=3.0)
    mf = mfcc(clip, n_mfcc=13)
    rows.append({
        "path": row["path"],
        "label": row["label"],
        "binary": "traffic" if row["label"] == "traffic" else "siren",
        "zcr_mean": float(np.mean(zero_crossing_rate(clip))),
        "centroid_mean": float(np.mean(spectral_centroid(clip))),
        "bandwidth_mean": float(np.mean(spectral_bandwidth(clip))),
        "rolloff_mean": float(np.mean(spectral_rolloff(clip))),
        **{f"mfcc_{i+1}_mean": float(np.mean(mf[i])) for i in range(min(13, mf.shape[0]))},
    })

feat = pd.DataFrame(rows)
if not feat.empty:
    feat.to_csv(TAB / "sirennet_feature_sample.csv", index=False)
    desc = feat.groupby("label")[["zcr_mean", "centroid_mean", "bandwidth_mean", "rolloff_mean"]].agg(["mean", "std"])
    desc.to_csv(TAB / "sirennet_descriptor_summary.csv")
feat.head() if not feat.empty else feat


,path,label,binary,zcr_mean,centroid_mean,bandwidth_mean,rolloff_mean,mfcc_1_mean,mfcc_2_mean,mfcc_3_mean,mfcc_4_mean,mfcc_5_mean,mfcc_6_mean,mfcc_7_mean,mfcc_8_mean,mfcc_9_mean,mfcc_10_mean,mfcc_11_mean,mfcc_12_mean,mfcc_13_mean
0,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,ambulance,siren,0.120742,2119.331167,2202.722315,3784.912482,-937.069908,182.524950,-88.394986,-6.151545,22.906682,9.654523,16.347269,33.974206,2.974515,8.614150,12.007022,-3.782923,-10.759718
1,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,ambulance,siren,0.109343,1776.179397,1922.344585,2578.559980,-1262.391075,220.780187,-78.313411,-104.180751,-18.000341,8.326323,7.238437,9.382495,-19.429943,-22.258545,11.763489,-8.240251,-12.379008
2,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,ambulance,siren,0.163732,2344.692221,2162.647533,4073.227278,-1091.377019,145.327831,-100.110921,23.214988,-0.658287,21.740430,33.302707,37.875946,-13.149082,1.222034,-4.003461,-7.936693,-23.926308
3,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,ambulance,siren,0.104175,1443.031488,930.069575,1951.631456,-1768.331547,301.988330,-280.324864,-103.575138,-46.387168,23.082076,17.216112,28.936644,-22.811532,-33.932302,-6.540904,3.197149,-7.480740
4,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,ambulance,siren,0.134347,1823.714960,1105.951214,2776.961325,-1947.454799,173.950696,-214.140642,20.927712,-18.380091,96.431533,105.630034,30.034235,-86.011651,-47.015044,77.679357,95.622689,-55.014366


## Descriptores espectrales por clase


In [5]:
if feat.empty:
    print("Sin características: monta sireNNet y re-ejecuta")
else:
    long = feat.melt(
        id_vars=["label", "binary"],
        value_vars=["zcr_mean", "centroid_mean", "bandwidth_mean", "rolloff_mean"],
        var_name="feature",
        value_name="value",
    )
    g = sns.catplot(data=long, x="label", y="value", col="feature", kind="box", sharey=False, height=3.2, aspect=0.95, color="#3b6d9a")
    g.set_xticklabels(rotation=35)
    g.savefig(FIG / "descriptors_by_class.png", bbox_inches="tight")
    plt.show()


/tmp/ipykernel_44298/3952155383.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## MFCC medios: sirena vs tráfico y entre tipos


In [6]:
if not feat.empty:
    mfcc_cols = [c for c in feat.columns if c.startswith("mfcc_")]
    means = feat.groupby("label")[mfcc_cols].mean()
    fig, ax = plt.subplots(figsize=(8, 4))
    for label, series in means.iterrows():
        ax.plot(range(1, len(series) + 1), series.values, marker="o", label=label)
    ax.set_xlabel("Coeficiente MFCC")
    ax.set_ylabel("Media en la muestra")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG / "mfcc_means_by_class.png", bbox_inches="tight")
    plt.show()
    means


/tmp/ipykernel_44298/3616100810.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Ejemplo log-mel vs MFCC

La CNN 2D opera de forma natural sobre el mapa log-mel (imagen tiempo-frecuencia). El MFCC comprime ese mapa a pocos coeficientes: útil para baselines clásicos, menos expresivo para redes convolucionales.


In [7]:
if sirennet.empty:
    print("sin ejemplo")
else:
    pick = {}
    for label, sub in sirennet.groupby("label"):
        pick[label] = sub.sample(1, random_state=SEED).iloc[0]["path"]
    labels = list(pick)
    fig, axes = plt.subplots(len(labels), 2, figsize=(10, 2.3 * len(labels)), squeeze=False)
    for i, label in enumerate(labels):
        clip = load_audio(pick[label], duration=3.0)
        mel = log_mel_spectrogram(clip)
        mf = mfcc(clip, n_mfcc=13)
        axes[i, 0].imshow(mel, origin="lower", aspect="auto", cmap="magma")
        axes[i, 0].set_ylabel(label)
        axes[i, 1].imshow(mf, origin="lower", aspect="auto", cmap="coolwarm")
        if i == 0:
            axes[i, 0].set_title("Log-mel")
            axes[i, 1].set_title("MFCC")
    fig.tight_layout()
    fig.savefig(FIG / "logmel_vs_mfcc_examples.png", bbox_inches="tight")
    plt.show()


/tmp/ipykernel_44298/4207304514.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Resultados

- **Detección:** ZCR, centroide y rolloff, más los MFCC bajos, son candidatos fuertes para sirena vs tráfico.
- **Tipo:** se espera más solapamiento; el mapa log-mel conserva modulación temporal (wail/yelp/hi-lo) que el vector MFCC promedio pierde.
- **Decisión de representación:** CNN sobre log-mel 2D como modelo principal; MFCC (13–20) como baseline tabular/SVM/CNN 1D.
- Siguiente fase: splits agrupados y entrenamiento, no más EDA.


In [8]:
result = {
    "seed": SEED,
    "librosa": HAS_LIBROSA,
    "n_feature_rows": int(len(feat)) if "feat" in globals() else 0,
    "figures": sorted(p.name for p in FIG.glob("*.png")),
}
result


{'seed': 7,
 'librosa': False,
 'n_feature_rows': 160,
 'figures': ['class_balance.png',
  'descriptors_by_class.png',
  'duration_hist.png',
  'logmel_vs_mfcc_examples.png',
  'mfcc_means_by_class.png',
  'sirennet_waveform_stft.png',
  'urbansound8k_class_counts.png']}